# Tema 14 — Seguimiento de objetos (tracking) con YOLO y BoT-SORT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-07/Tema-14/Tema_14.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En este notebook usamos un detector **YOLO** junto con el algoritmo de seguimiento **BoT-SORT** para detectar personas en un video y **asignarles un identificador (ID) estable** que se mantiene cuadro a cuadro, incluso ante oclusiones.

> ⚠️ Este notebook abre ventanas con `cv2.imshow`, por lo que está pensado para ejecutarse en **local** (VS Code / Jupyter) con acceso a pantalla. En Colab habría que adaptarlo para **guardar el video** en disco en lugar de mostrarlo en vivo. Necesitas el archivo de video de prueba (`MOT17 04 FRCNN raw.mp4`).

**Inicializar los parámetros y umbrales de seguimiento de objetos**

### ¿Qué hace este código?

Define la **configuración del algoritmo de seguimiento BoT-SORT** (normalmente guardada en un archivo `custom_tracker.yaml`). Cada parámetro controla cómo se asignan y conservan los IDs:

- **`track_high_thresh` / `track_low_thresh`**: umbrales de confianza para aceptar o recuperar detecciones.
- **`new_track_thresh`**: qué tan seguro debe estar el modelo para crear un **ID nuevo** (evita falsos positivos).
- **`track_buffer`**: cuántos cuadros "recuerda" a un objeto **oculto** antes de descartarlo (manejo de oclusiones).
- **`match_thresh`**: similitud necesaria para asociar una detección al mismo ID.
- **`with_reid` + `proximity_thresh` / `appearance_thresh`**: activan la **reidentificación visual**, comparando la apariencia para no cambiar de ID cuando un objeto reaparece.

In [ ]:
# Ultralytics AGPL-3.0 License
# BoT-SORT tracker defaults for mode="track"

tracker_type: botsort # Define el uso de características avanzadas de BoT-SORT
track_high_thresh: 0.5 # Umbral inicial; valores altos limpian los rastros falsos
track_low_thresh: 0.1 # Umbral secundario para recuperar detecciones débiles
new_track_thresh: 0.7 # Exigencia para iniciar un ID nuevo (evita falsos positivos)
track_buffer: 250 # Cantidad de frames que "recuerda" un objeto oculto (Oclusión)
match_thresh: 0.8 # Similitud requerida para asociar el mismo ID
fuse_score: True # Fusiona la confianza de detección con el movimiento

# BoT-SORT specifics
gmc_method: sift # Compensación de movimiento global (útil si la cámara se mueve)

# ReID model related thresh (Reidentificación visual)
proximity_thresh: 0.5 # Distancia máxima para considerar el ReID
appearance_thresh: 0.5 # Similitud visual requerida para no cambiar de ID
with_reid: True # Activa el uso de características visuales profundas
model: auto # Modelo automático para extraer características

**Seguimiento de objetos con asignación de cuadros delimitadores**

### ¿Qué hace este código?

Ejecuta el **bucle de seguimiento en tiempo real** sobre el video:

1. **Configura el dispositivo** (GPU si hay `cuda`, si no CPU) y carga el modelo **YOLO**.
2. Abre el video con **OpenCV** y, por cada cuadro (*frame*), hace un **recorte central** de 300×300 px para acelerar el procesamiento.
3. Llama a `model.track(...)` con `persist=True` para que el tracker (**BoT-SORT**, definido en `custom_tracker.yaml`) **mantenga los IDs** entre cuadros; se filtran solo personas (`classes=[0]`).
4. Para cada objeto confirmado dibuja su **bounding box**, un **color único derivado de su ID** y una etiqueta con el ID y la confianza.
5. Muestra el resultado en una ventana en vivo (`cv2.imshow`) y termina al presionar **`q`**, liberando los recursos.

In [ ]:
import cv2
import torch
from ultralytics import YOLO
# 1. Configuración de Dispositivo y Modelo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolo26x.pt').to(device)
# 2. Configuración de Video
video_path = 'MOT17 04 FRCNN raw.mp4'
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: No se pudo abrir el video.")
    exit()
# Configuraciones de detección y tracking
SCORE_THRESH = 0.7
IOU_THRESH = 0.6
TARGET_CLASSES = [0]  # Solo personas
print(f"Iniciando Tracking en {device}... Presiona 'q' para salir.")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    # 3. Recorte (Crop) Central para optimizar el procesamiento
    h_orig, w_orig = frame.shape[:2]
    crop_w, crop_h = 300, 300
    start_x = max(0, w_orig // 2 - (crop_w // 2))
    start_y = max(0, h_orig // 2 - (crop_h // 2))
    frame_crop = frame[start_y:start_y+crop_h, start_x:start_x+crop_w]
    # 4. Tracking
    results = model.track(
        source=frame_crop,
        persist=True,
        iou=IOU_THRESH,      # Umbral de Intersection over Union para asociación
        imgsz=400,          # Resolución de inferencia
        classes=TARGET_CLASSES, # Filtrar solo personas
        tracker="custom_tracker.yaml", # Archivo de configuración del algoritmo de tracking
        device=device,
        verbose=False,
    )[0]
    # 5. Proyección de resultados si hay identificadores confirmados
    if results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy().astype(int)
        ids = results.boxes.id.cpu().numpy().astype(int)
        confs = results.boxes.conf.cpu().numpy()

        for box, obj_id, conf in zip(boxes, ids, confs):
            if conf > SCORE_THRESH:
                xmin, ymin, xmax, ymax = box
                # --- Generación de color dinámico único basado en el ID ---
                color = (int((obj_id * 50) % 255), 255, int((obj_id * 30) % 255))
                # Dibujar Rectángulo y metadatos (ID y Confianza)
                cv2.rectangle(frame_crop, (xmin, ymin), (xmax, ymax), color, 2)
                display_text = f"ID:{obj_id} Pers: {conf:.2f}"
                cv2.putText(frame_crop, display_text, (xmin, ymin - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    # 6. Despliegue en pantalla
    cv2.imshow('Real-Time Tracking', frame_crop)
    if cv2.waitKey(1) & 0xFF == ord('q'): break
cap.release()
cv2.destroyAllWindows()